<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/08_sandbox_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. The Sandbox Setup
# ---------------------------------------------------------
# Let's say the distance between 4-bit quantization buckets is 0.133
BUCKET_SIZE = 0.133

# We create a dummy model with just ONE weight, starting at 0.50
class DummyNeuron(nn.Module):
    def __init__(self):
        super().__init__()
        # The weight starts at 0.50
        self.weight = nn.Parameter(torch.tensor([0.50]))

    def forward(self, x):
        return self.weight * x

def simulate_4bit_quantization(weight_tensor):
    """Rounds the weight to the nearest 4-bit bucket."""
    return torch.round(weight_tensor / BUCKET_SIZE) * BUCKET_SIZE

# ---------------------------------------------------------
# 2. Standard Unlearning Loop (The Baseline Failure)
# ---------------------------------------------------------
standard_model = DummyNeuron()
# A tiny learning rate, typical for LLM fine-tuning
standard_optimizer = torch.optim.SGD(standard_model.parameters(), lr=0.01)

print("--- Running Standard Unlearning ---")
for epoch in range(5):
    standard_optimizer.zero_grad()

    # We simulate an unlearning loss that gently pushes the weight downward
    loss = standard_model.weight
    loss.backward()
    standard_optimizer.step()

    current_w = standard_model.weight.item()
    quantized_w = simulate_4bit_quantization(standard_model.weight).item()
    print(f"Epoch {epoch+1} | 16-bit Weight: {current_w:.4f} | Quantized (Edge) Weight: {quantized_w:.4f}")

# ---------------------------------------------------------
# 3. Grid-Aware Unlearning Loop (Your Framework)
# ---------------------------------------------------------
grid_model = DummyNeuron()
grid_optimizer = torch.optim.SGD(grid_model.parameters(), lr=0.01)
original_weight = grid_model.weight.clone().detach()

print("\n--- Running Grid-Aware Unlearning ---")
for epoch in range(5):
    grid_optimizer.zero_grad()

    # 1. The gentle downward push (standard unlearning)
    base_loss = grid_model.weight

    # 2. YOUR NOVELTY: The Grid Penalty
    weight_diff = torch.abs(grid_model.weight - original_weight)

    # If the weight hasn't moved further than BUCKET_SIZE, apply a massive penalty
    # We multiply by 10 (lambda) to enforce the penalty strongly
    grid_penalty = torch.relu(BUCKET_SIZE - weight_diff) * 10.0

    # Combine losses
    total_loss = base_loss + grid_penalty

    total_loss.backward()
    grid_optimizer.step()

    current_w = grid_model.weight.item()
    quantized_w = simulate_4bit_quantization(grid_model.weight).item()
    print(f"Epoch {epoch+1} | 16-bit Weight: {current_w:.4f} | Quantized (Edge) Weight: {quantized_w:.4f}")

--- Running Standard Unlearning ---
Epoch 1 | 16-bit Weight: 0.4900 | Quantized (Edge) Weight: 0.5320
Epoch 2 | 16-bit Weight: 0.4800 | Quantized (Edge) Weight: 0.5320
Epoch 3 | 16-bit Weight: 0.4700 | Quantized (Edge) Weight: 0.5320
Epoch 4 | 16-bit Weight: 0.4600 | Quantized (Edge) Weight: 0.3990
Epoch 5 | 16-bit Weight: 0.4500 | Quantized (Edge) Weight: 0.3990

--- Running Grid-Aware Unlearning ---
Epoch 1 | 16-bit Weight: 0.4900 | Quantized (Edge) Weight: 0.5320
Epoch 2 | 16-bit Weight: 0.3800 | Quantized (Edge) Weight: 0.3990
Epoch 3 | 16-bit Weight: 0.2700 | Quantized (Edge) Weight: 0.2660
Epoch 4 | 16-bit Weight: 0.2600 | Quantized (Edge) Weight: 0.2660
Epoch 5 | 16-bit Weight: 0.2500 | Quantized (Edge) Weight: 0.2660
